# Banc d’essai AP-HP : sélection de scénarios et génération en une ou deux étapes

Ce notebook permet de :

1. installer automatiquement `fictomed` depuis la branche Git `prompt-work` ;
2. filtrer `data/aphp/scenarios_bn_all_20260128.pq` ;
3. générer les scénarios complets avec `fictomed` ;
4. créer des fichiers de prompts éditables ;
5. lancer Mistral en une génération directe ou en deux étapes : résumé clinique puis CR.

Arborescence créée pour `RUN_NAME = "test_xx"` :

```text
test_xx/
├── scenarios_fictomed_selected.parquet
├── manifest.parquet
├── prompts/une_gen/
│   ├── system_prompt/
│   └── user_prompt/
├── prompts/deux_gen/
│   ├── premiere_gen/
│   │   ├── system_prompt/
│   │   └── user_prompt/
│   └── deuxieme_gen/
│       ├── system_prompt/
│       └── user_prompt/
└── sorties/
    ├── CR_1gen/
    └── resume_CR_2gen/
```

Les prompts sont écrits dans des fichiers `.txt` modifiables manuellement avant les appels Mistral. Le dossier de travail est créé à côté du notebook par défaut.


## 0. Chargement du script utilitaire et localisation du projet

La racine Stream est détectée automatiquement. En cas d’échec, renseigner `PROJECT_ROOT_OVERRIDE`.

In [2]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import importlib
import os
import sys

import polars as pl
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
UTILS_SCRIPT = NOTEBOOK_DIR / "aphp_generation_utils.py"

if not UTILS_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Script utilitaire introuvable : {UTILS_SCRIPT}\n"
        "Placez aphp_generation_utils.py dans le même dossier que le notebook."
    )

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import aphp_generation_utils as utils
importlib.reload(utils)

# Laisser None pour la détection automatique.
# Exemple : PROJECT_ROOT_OVERRIDE = Path("/chemin/vers/Stream")
PROJECT_ROOT_OVERRIDE: Path | None = None

PROJECT_ROOT = (
    utils.find_stream_root(NOTEBOOK_DIR)
    if PROJECT_ROOT_OVERRIDE is None
    else Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
)

if not utils.looks_like_stream_root(PROJECT_ROOT):
    raise FileNotFoundError(
        f"Le chemin ne ressemble pas à une racine Stream : {PROJECT_ROOT}"
    )

BASE_DIR = NOTEBOOK_DIR
DEPENDENCIES_DIR = NOTEBOOK_DIR / "_dependencies"
FICTOMED_LOCAL_DIR = DEPENDENCIES_DIR / "fictomed_prompt_work"
FICTOMED_REPO_URL = "https://github.com/CHU-Brest/fictomed.git"
FICTOMED_BRANCH = "prompt-work"

print("Python du kernel :", sys.executable)
print("Notebook         :", NOTEBOOK_DIR)
print("Script utilitaire:", UTILS_SCRIPT)
print("Projet Stream    :", PROJECT_ROOT)
print("Dépendances      :", DEPENDENCIES_DIR)
print("Dépôt fictomed   :", FICTOMED_LOCAL_DIR)

Python du kernel : /Users/remi/Documents/Stream/.venv/bin/python
Notebook         : /Users/remi/Documents/Stream/work_modif_prompts
Script utilitaire: /Users/remi/Documents/Stream/work_modif_prompts/aphp_generation_utils.py
Projet Stream    : /Users/remi/Documents/Stream
Dépendances      : /Users/remi/Documents/Stream/work_modif_prompts/_dependencies
Dépôt fictomed   : /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work


## 1. Installation ou mise à jour de fictomed

Cette cellule clone la branche configurée, met à jour le dépôt lorsqu’il n’existe aucune modification locale, puis effectue une installation éditable dans le Python du kernel.

In [3]:
utils.install_or_update_fictomed(
    dependencies_dir=DEPENDENCIES_DIR,
    local_dir=FICTOMED_LOCAL_DIR,
    repo_url=FICTOMED_REPO_URL,
    branch=FICTOMED_BRANCH,
    project_root=PROJECT_ROOT,
    python_executable=sys.executable,
)

FICTOMED_INSTALLATION = utils.verify_fictomed_installation(
    local_dir=FICTOMED_LOCAL_DIR,
    expected_branch=FICTOMED_BRANCH,
)


$ git ls-remote --heads https://github.com/CHU-Brest/fictomed.git refs/heads/prompt-work
83ebca9ac9b38327a7ed634de420f9c225172769	refs/heads/prompt-work


$ git clone --branch prompt-work --single-branch https://github.com/CHU-Brest/fictomed.git /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work


Cloning into '/Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work'...



$ /Users/remi/Documents/Stream/.venv/bin/python -m pip install --editable /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work
Obtaining file:///Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for fictomed (pyproject.toml): started
  Building editable for fictomed (pyproject.toml): finished with status 'done'
  Created wheel for fictomed: filename=fictomed-0.1.0-0.editable-py3-none-any.whl size=10945 sha256=3a749406

## 2. Configuration générale du run

In [10]:
RUN_NAME = "test_xx"
RUN_DIR = BASE_DIR / RUN_NAME

SOURCE_PROFILES_PATH = (
    PROJECT_ROOT / "data/aphp/scenarios_bn_all_20260128.pq"
)
APHP_DATA_DIR = PROJECT_ROOT / "data/aphp"

ONE_GEN_TEMPLATES_DIR = NOTEBOOK_DIR / "template_one_unique_gen"
FIRST_GEN_TEMPLATES_DIR = NOTEBOOK_DIR / "template_first_gen"
SECOND_GEN_TEMPLATES_DIR = NOTEBOOK_DIR / "template_second_gen"

TARGET_N = 1
CANDIDATE_POOL_SIZE = 200
RANDOM_SELECTION = True
RANDOM_SEED = 42

# True supprime les anciens .txt des six dossiers de prompts avant recréation.
# Passer à False dès que les prompts ont été modifiés manuellement.
CLEAN_EXISTING_PROMPTS = True

FICTOMED_CONFIG_FILE = NOTEBOOK_DIR / "data/servers.yaml"
PATHS = utils.create_run_tree(RUN_DIR)

print("BASE_DIR                :", BASE_DIR)
print("PROJECT_ROOT            :", PROJECT_ROOT)
print("RUN_DIR                 :", RUN_DIR)
print("SOURCE_PROFILES_PATH    :", SOURCE_PROFILES_PATH)
print("APHP_DATA_DIR           :", APHP_DATA_DIR)
print("ONE_GEN_TEMPLATES_DIR   :", ONE_GEN_TEMPLATES_DIR)
print("FIRST_GEN_TEMPLATES_DIR :", FIRST_GEN_TEMPLATES_DIR)
print("SECOND_GEN_TEMPLATES_DIR:", SECOND_GEN_TEMPLATES_DIR)
print("FICTOMED_CONFIG_FILE    :", FICTOMED_CONFIG_FILE)
print("CLEAN_EXISTING_PROMPTS  :", CLEAN_EXISTING_PROMPTS)

for key, value in PATHS.items():
    print(f"{key:>14}: {value}")

if not SOURCE_PROFILES_PATH.exists():
    raise FileNotFoundError(
        f"Fichier source AP-HP introuvable : {SOURCE_PROFILES_PATH}"
    )

BASE_DIR                : /Users/remi/Documents/Stream/work_modif_prompts
PROJECT_ROOT            : /Users/remi/Documents/Stream
RUN_DIR                 : /Users/remi/Documents/Stream/work_modif_prompts/test_xx
SOURCE_PROFILES_PATH    : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq
APHP_DATA_DIR           : /Users/remi/Documents/Stream/data/aphp
ONE_GEN_TEMPLATES_DIR   : /Users/remi/Documents/Stream/work_modif_prompts/template_one_unique_gen
FIRST_GEN_TEMPLATES_DIR : /Users/remi/Documents/Stream/work_modif_prompts/template_first_gen
SECOND_GEN_TEMPLATES_DIR: /Users/remi/Documents/Stream/work_modif_prompts/template_second_gen
FICTOMED_CONFIG_FILE    : /Users/remi/Documents/Stream/work_modif_prompts/data/servers.yaml
CLEAN_EXISTING_PROMPTS  : True
    one_system: /Users/remi/Documents/Stream/work_modif_prompts/test_xx/prompts/une_gen/system_prompt
      one_user: /Users/remi/Documents/Stream/work_modif_prompts/test_xx/prompts/une_gen/user_prompt
  first_system: /Use

## 3. Filtres

Tous les filtres d’une liste sont cumulatifs. Opérateurs disponibles : `eq`, `ne`, `in`, `not_in`, `startswith`, `endswith`, `contains`, `regex`, `gt`, `ge`, `lt`, `le`, `is_null`, `not_null`. `exclude=True` inverse une condition.

In [11]:
SOURCE_FILTERS: list[dict[str, Any]] = [
    {"column": "diag2", "op": "endswith", "value": "8"},
    # {"column": "diag2", "op": "endswith", "value": "9", "exclude": True},
]

SCENARIO_FILTERS: list[dict[str, Any]] = [
    # {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]

print("Filtres source   :", SOURCE_FILTERS)
print("Filtres scénario :", SCENARIO_FILTERS)

Filtres source   : [{'column': 'diag2', 'op': 'endswith', 'value': '8'}]
Filtres scénario : []


## 4. Lecture, filtrage et échantillonnage des profils source

In [12]:
source_path, source_df, source_filtered, candidate_source = (
    utils.prepare_source_candidates(
        source_profiles_path=SOURCE_PROFILES_PATH,
        source_filters=SOURCE_FILTERS,
        candidate_pool_size=CANDIDATE_POOL_SIZE,
        random_selection=RANDOM_SELECTION,
        random_seed=RANDOM_SEED,
    )
)

SOURCE_PREVIEW_COLUMNS = [
    "source_row_id",
    "source_scenario_id",
    "mode_hospit",
    "racine",
    "ghm2",
    "diag2",
    "mdp",
    "diagnostic_associes",
    "duree",
]
SOURCE_PREVIEW_COLUMNS = [
    column
    for column in SOURCE_PREVIEW_COLUMNS
    if column in candidate_source.columns
]

display(candidate_source.select(SOURCE_PREVIEW_COLUMNS).head(20))

Fichier source : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq
Shape          : (142912, 14)
Colonnes       : ['mode_hospit', 'sexe', 'age', 'racine', 'ghm2', 'diag2', 'mdp', 'nbda', 'diagnostic_associes', 'n', 'mode_entree', 'mode_sortie', 'agean', 'duree']
SOURCE, filtre 1: {'column': 'diag2', 'op': 'endswith', 'value': '8'} | 142912 -> 7716

Candidats envoyés à fictomed : (200, 16)


source_row_id,source_scenario_id,mode_hospit,racine,ghm2,diag2,mdp,diagnostic_associes,duree
u32,str,str,str,str,str,str,str,f64
115709,"""scenarios_bn_all_20260128_row_…","""HC""","""13M04""","""13M041""","""N938""","""DP""","""D500""",2.0
45723,"""scenarios_bn_all_20260128_row_…","""HC""","""01M21""","""01M211""","""R5218""","""DP""","""M7978""",3.0
140790,"""scenarios_bn_all_20260128_row_…","""HC""","""15M06""","""15M06A""","""P598""","""DP""","""Z1351""",6.0
100057,"""scenarios_bn_all_20260128_row_…","""HC""","""23M20""","""23M20Z""","""G258""","""Z04801""","""G473""",3.0
112365,"""scenarios_bn_all_20260128_row_…","""HC""","""14Z14""","""14Z14A""","""O48""","""DP""","""O244 O700 Z370 Z391""",6.0
…,…,…,…,…,…,…,…,…
78011,"""scenarios_bn_all_20260128_row_…","""HP""","""14M03""","""14M03T""","""Z358""","""DP""","""O244 Z359 Z713""",0.0
30292,"""scenarios_bn_all_20260128_row_…","""HC""","""23K02""","""23K02Z""","""Z468""","""DP""","""J961 Z991""",1.0
6983,"""scenarios_bn_all_20260128_row_…","""HC""","""14M03""","""14M03A""","""Z358""","""DP""","""O441""",2.0


## 5. Création du fichier `servers.yaml` utilisé par fictomed

In [13]:
FICTOMED_CONFIG_PATHS = utils.write_fictomed_config(
    config_file=FICTOMED_CONFIG_FILE,
    project_root=PROJECT_ROOT,
    run_dir=RUN_DIR,
)

print("Input AP-HP configuré :", FICTOMED_CONFIG_PATHS["aphp_input"])
print("Sortie fictomed       :", FICTOMED_CONFIG_PATHS["aphp_output"])
print("Référentiels          :", FICTOMED_CONFIG_PATHS["aphp_referentials"])

Configuration fictomed : /Users/remi/Documents/Stream/work_modif_prompts/data/servers.yaml

pipelines:
  brest:
    data:
      input: /Users/remi/Documents/Stream/data/brest
      output: /Users/remi/Documents/Stream/work_modif_prompts/test_xx/sorties/scenarios_brest
  aphp:
    data:
      input: /Users/remi/Documents/Stream/data/aphp
      output: /Users/remi/Documents/Stream/work_modif_prompts/test_xx/sorties/scenarios
      referentials: /Users/remi/Documents/Stream/data/aphp/referentials

Input AP-HP configuré : /Users/remi/Documents/Stream/data/aphp
Sortie fictomed       : /Users/remi/Documents/Stream/work_modif_prompts/test_xx/sorties/scenarios
Référentiels          : /Users/remi/Documents/Stream/data/aphp/referentials


## 6. Génération fictomed et sélection finale

Le fichier `profiles` résolu dans le dossier d’entrée AP-HP est sauvegardé, remplacé temporairement par les candidats filtrés, puis restauré dans un bloc `finally`.

In [14]:
generated_candidates, generated_filtered, selected_scenarios = (
    utils.generate_and_select_fictomed_scenarios(
        candidate_source=candidate_source,
        config_file=FICTOMED_CONFIG_FILE,
        aphp_data_dir=APHP_DATA_DIR,
        paths=PATHS,
        scenario_filters=SCENARIO_FILTERS,
        target_n=TARGET_N,
        random_selection=RANDOM_SELECTION,
        random_seed=RANDOM_SEED,
        run_dir=RUN_DIR,
    )
)

SCENARIO_DISPLAY_COLUMNS = [
    "generation_id",
    "source_row_id",
    "source_scenario_id",
    "icd_primary_code",
    "drg_parent_code",
    "ghm2",
    "template_name",
    "coding_rule",
    "admission_type",
]
SCENARIO_DISPLAY_COLUMNS = [
    column
    for column in SCENARIO_DISPLAY_COLUMNS
    if column in selected_scenarios.columns
]

display(selected_scenarios.select(SCENARIO_DISPLAY_COLUMNS))

fictomed importé depuis : /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work/fictomed/__init__.py
Dossier lu par fictomed : /Users/remi/Documents/Stream/data/aphp
Profiles actif          : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq
Backup                  : /Users/remi/Documents/Stream/work_modif_prompts/test_xx/_backups/scenarios_bn_all_20260128.pq.20260813_115417.backup
Vérification des données d'entrée pour le pipeline AP-HP.
Chargement des fichiers PMSI depuis /Users/remi/Documents/Stream/data/aphp.
Fichiers PMSI chargés avec succès.
Données AP-HP présentes dans /Users/remi/Documents/Stream/data/aphp.
















































Construction contexte:   0%|          | 0/3 [03:49<?, ?étape/s]

Scénarios sauvegardés dans /Users/remi/Documents/Stream/work_modif_prompts/test_xx/sorties/scenarios/aphp_scenarios_194_20260813_115422.parquet
Profiles original restauré : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq

Scénarios candidats générés : (194, 55)
source_row_id: 194 uniques / 194
source_scenario_id: 194 uniques / 194
generation_id: 194 uniques / 194

Scénarios retenus : (1, 55)
Écrit             : /Users/remi/Documents/Stream/work_modif_prompts/test_xx/scenarios_fictomed_selected.parquet


generation_id,source_row_id,source_scenario_id,icd_primary_code,drg_parent_code,ghm2,template_name,coding_rule,admission_type
str,i64,str,str,str,str,str,str,str
"""0c3482a6-95e4-4090-b06f-6520c9…",88260,"""scenarios_bn_all_20260128_row_…","""D508""","""16M11""","""16M111""","""medical_inpatient.txt""","""other""","""HC"""


## 7. Création des prompts et du manifest

Les trois `user_prompt` proviennent du scénario fictomed. Les trois `system_prompt` proviennent des dossiers de templates externes correspondant au `template_name`.

In [15]:
manifest = utils.create_prompt_files(
    selected_scenarios=selected_scenarios,
    run_dir=RUN_DIR,
    paths=PATHS,
    one_gen_templates_dir=ONE_GEN_TEMPLATES_DIR,
    first_gen_templates_dir=FIRST_GEN_TEMPLATES_DIR,
    second_gen_templates_dir=SECOND_GEN_TEMPLATES_DIR,
    clean_existing_prompts=CLEAN_EXISTING_PROMPTS,
)

display(manifest)

Anciens fichiers de prompts supprimés : 0
Manifest écrit : /Users/remi/Documents/Stream/work_modif_prompts/test_xx/manifest.parquet


generation_id,template_name,file_stem,one_system_path,one_user_path,first_system_path,first_user_path,second_system_path,second_user_path
str,str,str,str,str,str,str,str,str
"""0c3482a6-95e4-4090-b06f-6520c9…","""medical_inpatient.txt""","""0000__0c3482a6-95e4-4090-b06f-…","""prompts/une_gen/system_prompt/…","""prompts/une_gen/user_prompt/00…","""prompts/deux_gen/premiere_gen/…","""prompts/deux_gen/premiere_gen/…","""prompts/deux_gen/deuxieme_gen/…","""prompts/deux_gen/deuxieme_gen/…"


## 8. Prévisualisation des prompts

Après cette cellule, les fichiers `.txt` peuvent être modifiés manuellement. Ne pas recréer les prompts avec `CLEAN_EXISTING_PROMPTS=True` après ces modifications.

In [16]:
PROMPT_PREVIEW_INDEX = 0

utils.preview_prompt_files(
    run_dir=RUN_DIR,
    preview_index=PROMPT_PREVIEW_INDEX,
    max_chars=8_000,
)

### Scénario `0c3482a6-95e4-4090-b06f-6520c958327d` — `medical_inpatient.txt`

#### Une génération — system prompt

Vous êtes un médecin clinicien expert. Votre tâche est de générer un compte rendu d'hospitalisation détaillé à partir d'un scénario clinique réalisé avec des codes de la classification internationale des maladies (CIM-10) et d'autres informations décrivant l'hospitalisation.


# Contexte : le codage CIM-10

La CIM-10 est une classification des maladies, elle peut se définir comme un ensemble organisé de rubriques dans lesquelles on range des entités morbides en fonction de certains critères établis. La CIM est utilisée pour transposer les diagnostics de maladies ou autres problèmes de santé en codes alphanumériques, ce qui facilite le stockage, la recherche et l'analyse des données. Elle est très utilisée en France, en particulier pour le codage des causes de décès et pour la déclaration de l'activité hospitalière dans le cadre du programme de médicalisation des systèmes d'information (PMSI).

Le codage CIM-10 consiste à identifier dans un document médical (CRH, notes d'évolution, comp

#### Une génération — user prompt

**SCÉNARIO DE DÉPART :**
- Âge du patient : 83 ans
- Sexe du patient : Masculin
- Date d'entrée : 18/11/2024
- Date de sortie : 21/11/2024
- Date de naissance : 10/09/1941
- Nom du patient : Neves cordeiro
- Prénom du patient : Jonathann
- Mode d'entrée' : DOMICILE
- Mode de sortie' : DOMICILE
- Contexte de l'hospitalisation : Pour prise en charge diagnostique et thérapeutique du diagnotic principal en hospitalisation complète.
- Codage CIM10 :
   * Diagnostic principal : Autres anémies par carence en fer (D508)
   * Diagnostics associés :
      - Hypertension essentielle (primitive) (I10)
- Nom du médecin / signataire : Genevieve Mainardi
- Service : MEDECINE INTERNE
- Hôpital : Hôpital Claude Huriez - Centre Hospitalier Universitaire de Lille

**FICHES DESCRIPTIVES DES CODES CIM-10 DU SCÉNARIO :**

Les fiches ci-dessous précisent le périmètre des codes CIM-10 présents dans le scénario.
Elles doivent être utilisées pour choisir des formulations compatibles avec les codes, sans ajouter

...

**SCÉNARIO DE DÉPART :**
- Âge du patient : 83 ans
- Sexe du patient : Masculin
- Date d'entrée : 18/11/2024
- Date de sortie : 21/11/2024
- Date de naissance : 10/09/1941
- Nom du patient : Neves cordeiro
- Prénom du patient : Jonathann
- Mode d'entrée' : DOMICILE
- Mode de sortie' : DOMICILE
- Contexte de l'hospitalisation : Pour prise en charge diagnostique et thérapeutique du diagnotic principal en hospitalisation complète.
- Codage CIM10 :
   * Diagnostic principal : Autres anémies par carence en fer (D508)
   * Diagnostics associés :
      - Hypertension essentielle (primitive) (I10)
- Nom du médecin / signataire : Genevieve Mainardi
- Service : MEDECINE INTERNE
- Hôpital : Hôpital Claude Huriez - Centre Hospitalier Universitaire de Lille

**FICHES DESCRIPTIVES DES CODES CIM-10 DU SCÉNARIO :**

Les fiches ci-dessous précisent le périmètre des codes CIM-10 présents dans le scénario.
Elles doivent être utilisées pour choisir des formulations compatibles avec les codes, sans ajouter

#### Deux générations, deuxième — system prompt

Vous êtes un médecin clinicien expert.

Pour cette deuxième génération, votre tâche est de rédiger les documents médicaux finaux à partir du scénario clinique complet et du résumé clinique contrôlé généré lors du premier appel.

Le résumé est fourni en plus du scénario, des fiches descriptives, des actes éventuels et des informations administratives. Il ne remplace aucune de ces données.

# Contexte : le codage CIM-10

La CIM-10 est une classification des maladies, elle peut se définir comme un ensemble organisé de rubriques dans lesquelles on range des entités morbides en fonction de certains critères établis. La CIM est utilisée pour transposer les diagnostics de maladies ou autres problèmes de santé en codes alphanumériques, ce qui facilite le stockage, la recherche et l'analyse des données. Elle est très utilisée en France, en particulier pour le codage des causes de décès et pour la déclaration de l'activité hospitalière dans le cadre du programme de médicalisation des systèmes d'

#### Deux générations, deuxième — user prompt

**SCÉNARIO DE DÉPART :**
- Âge du patient : 83 ans
- Sexe du patient : Masculin
- Date d'entrée : 18/11/2024
- Date de sortie : 21/11/2024
- Date de naissance : 10/09/1941
- Nom du patient : Neves cordeiro
- Prénom du patient : Jonathann
- Mode d'entrée' : DOMICILE
- Mode de sortie' : DOMICILE
- Contexte de l'hospitalisation : Pour prise en charge diagnostique et thérapeutique du diagnotic principal en hospitalisation complète.
- Codage CIM10 :
   * Diagnostic principal : Autres anémies par carence en fer (D508)
   * Diagnostics associés :
      - Hypertension essentielle (primitive) (I10)
- Nom du médecin / signataire : Genevieve Mainardi
- Service : MEDECINE INTERNE
- Hôpital : Hôpital Claude Huriez - Centre Hospitalier Universitaire de Lille

**FICHES DESCRIPTIVES DES CODES CIM-10 DU SCÉNARIO :**

Les fiches ci-dessous précisent le périmètre des codes CIM-10 présents dans le scénario.
Elles doivent être utilisées pour choisir des formulations compatibles avec les codes, sans ajouter

## 9. Configuration Mistral

`GENERATION_MODE` peut valoir `none`, `one`, `two` ou `both`. Les tarifs sont des variables de configuration : les mettre à jour en même temps que le modèle lorsque nécessaire.

In [ ]:
GENERATION_MODE = "two"  # "none", "one", "two" ou "both"
MODEL = os.getenv("MISTRAL_MODEL", "mistral-large-latest")

MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
POLL_INTERVAL_SECONDS = 1

USE_ORIGINAL_PREFIX_ONE_GEN = True
USE_ORIGINAL_PREFIX_SECOND_GEN = True
FIRST_GEN_PREFIX = "Résumé clinique :"

SUMMARY_INSERT_HEADER = """
### RÉSUMÉ CLINIQUE ISSU DE LA PREMIÈRE GÉNÉRATION
Le résumé ci-dessous est une aide intermédiaire.
Le scénario clinique, les codes, les fiches descriptives et les instructions
restent prioritaires en cas de divergence.
"""

SUMMARY_INSERT_FOOTER = """
### FIN DU RÉSUMÉ CLINIQUE INTERMÉDIAIRE
"""

MISTRAL_BATCH_INPUT_USD_PER_MILLION = 0.25
MISTRAL_BATCH_OUTPUT_USD_PER_MILLION = 0.75

MISTRAL_API_KEY = "xxx" ### A COMPLETER

print("Mode                         :", GENERATION_MODE)
print("Modèle                       :", MODEL)
print("Clé Mistral disponible       :", bool(MISTRAL_API_KEY))
print("MAX_TOKENS_SUMMARY           :", MAX_TOKENS_SUMMARY)
print("MAX_TOKENS_CR                :", MAX_TOKENS_CR)
print(
    "Tarif Batch entrée / 1M tok :",
    f"${MISTRAL_BATCH_INPUT_USD_PER_MILLION:.2f}",
)
print(
    "Tarif Batch sortie / 1M tok :",
    f"${MISTRAL_BATCH_OUTPUT_USD_PER_MILLION:.2f}",
)

Mode                         : two
Modèle                       : mistral-large-latest
Clé Mistral disponible       : True
MAX_TOKENS_SUMMARY           : 8000
MAX_TOKENS_CR                : 128000
Tarif Batch entrée / 1M tok : $0.25
Tarif Batch sortie / 1M tok : $0.75


## 10. Exécution Mistral et bilan tokens/coûts

In [13]:
GENERATION_RESULTS = utils.run_generation_workflow(
    generation_mode=GENERATION_MODE,
    api_key=MISTRAL_API_KEY,
    run_dir=RUN_DIR,
    paths=PATHS,
    model=MODEL,
    max_tokens_summary=MAX_TOKENS_SUMMARY,
    max_tokens_cr=MAX_TOKENS_CR,
    poll_interval_seconds=POLL_INTERVAL_SECONDS,
    use_original_prefix_one_gen=USE_ORIGINAL_PREFIX_ONE_GEN,
    use_original_prefix_second_gen=USE_ORIGINAL_PREFIX_SECOND_GEN,
    first_gen_prefix=FIRST_GEN_PREFIX,
    summary_insert_header=SUMMARY_INSERT_HEADER,
    summary_insert_footer=SUMMARY_INSERT_FOOTER,
    batch_input_usd_per_million=(
        MISTRAL_BATCH_INPUT_USD_PER_MILLION
    ),
    batch_output_usd_per_million=(
        MISTRAL_BATCH_OUTPUT_USD_PER_MILLION
    ),
)

print("Mode exécuté :", GENERATION_RESULTS["mode"])
print("Bilans usage :", list(GENERATION_RESULTS["usage"]))

if "usage_output_path" in GENERATION_RESULTS:
    print("Bilan JSON    :", GENERATION_RESULTS["usage_output_path"])


Lancement batch Mistral
Modèle                   : mistral-large-latest
Nombre de requêtes       : 1
Limite tokens de sortie  : 8000
Fichier JSONL d'entrée   : /home/cdechaux/Documents/AOC/codes/Stream/work_modif_prompts/test_xx/_mistral_batches/two_gen_first/batch_input_20260806_133356_810721.jsonl
Batch job ID             : 2be7b148-bf70-4536-b627-73da94a8fdf9
Statut : SUCCESS — 1/1
Statut final             : SUCCESS
Résultats JSONL          : /home/cdechaux/Documents/AOC/codes/Stream/work_modif_prompts/test_xx/_mistral_batches/two_gen_first/batch_output_20260806_133356_810721.jsonl
OK — première génération / résumés: 1 sortie(s)

PREMIÈRE GÉNÉRATION — RÉSUMÉS
Nombre de requêtes       : 1
Tokens d'entrée          : 7 757
Tokens de sortie         : 2 801
Tokens totaux            : 10 558
Moyenne / requête        : 10 558
Coût entrée estimé       : $0.001939
Coût sortie estimé       : $0.002101
COÛT TOTAL ESTIMÉ        : $0.004040

Lancement batch Mistral
Modèle                   : mi

## 11. Visualisation d’un scénario et de ses sorties

In [14]:
VIEW_INDEX = 0

utils.visualize_run_outputs(
    run_dir=RUN_DIR,
    paths=PATHS,
    view_index=VIEW_INDEX,
    scenario_max_chars=12_000,
    output_max_chars=20_000,
)

# Visualisation
- `generation_id` : `6a1a14c5-67b5-4adc-9ac2-94b39b2a9127`
- `template_name` : `medical_inpatient.txt`
- `DP` : `I638`
- `racine GHM` : `01M30`

## Scénario

**SCÉNARIO DE DÉPART :**
- Âge du patient : 72 ans
- Sexe du patient : Masculin
- Date d'entrée : 01/05/2025
- Date de sortie : 18/05/2025
- Date de naissance : 25/02/1953
- Nom du patient : Guery
- Prénom du patient : Nicolas
- Mode d'entrée' : DOMICILE
- Mode de sortie' : DOMICILE
- Contexte de l'hospitalisation : Pour prise en charge diagnostique et thérapeutique du diagnotic principal en hospitalisation complète.
- Codage CIM10 :
   * Diagnostic principal : Autres infarctus cérébraux (I638)
   * Diagnostics associés :
      - Ataxie, sans précision (R270)
      - Dysarthrie et anarthrie (R471)
      - Diabète sucré de type 2 non insulinotraité ou sans précision, sans complication (E1198)
      - Autres avitaminoses précisées du groupe B (E538)
      - Trouble anxieux et dépressif mixte (F412)
      - Diplopie (H532)
      - Nystagmus et autres anomalies des mouvements oculaires (H55)
      - Antécédents personnels de non-observance d'un traitement médical et d'un régime (Z911)
- No

## Résumé — première génération

Résumé clinique : voici le JSON strictement conforme à la demande.

```json
{
  "resume_clinique": [
    {
      "code_CIM10": "I63.8",
      "definition_code": "Autres infarctus cérébraux",
      "formulations_cliniques_autorisees": [
        "Infarctus cérébral de type précisé non classé ailleurs",
        "Ramollissement cérébral par artériosclérose",
        "Syndrome artériel cérébelleux supérieur"
      ],
      "elements_cliniques_autorises": "Occlusion ou sténose des artères cérébrales ou précérébrales entraînant un infarctus cérébral, avec ou sans mention d'hypertension associée. Localisation et mécanisme non précisés.",
      "prise_en_charge_pendant_le_sejour": "Prise en charge diagnostique (imagerie cérébrale, bilan étiologique vasculaire) et thérapeutique (traitement antiagrégant ou anticoagulant selon étiologie, contrôle des facteurs de risque vasculaire, rééducation neurologique). Surveillance des complications neurologiques.",
      "elements_a_ne_pas_ajouter": [
      

## CR — deuxième génération

Le compte rendu suivant respecte les élements suivants :
        - les diagnostics ont une formulation moins formelle que la définition du code
        - le plan du CRH est conforme aux recommandations. Les sections sont détaillées et cohérentes entre elles.
        - les dates sont plausibles et cohérentes
        - les informations obligatoires sont présentes
        - les informations sur le cancer sont détaillées si pertinent
        - les informations sont exhaustives et précises

```json
{
  "CR": "### Hôpital Pitié-Salpêtrière - Assistance Publique Hôpitaux de Paris
Service NEURO-VASCULAIRE

**Patient :**
Nom : Guery
Prénom : Nicolas
Date de naissance : 25/02/1953
Âge : 72 ans

---

### Motif d’hospitalisation
M. Nicolas Guery, 72 ans, a été admis le 01/05/2025 pour exploration et prise en charge d’un déficit neurologique d’installation brutale associant des troubles de la coordination, des difficultés d’élocution et une vision double. Ces symptômes ont motivé une hospitalisatio

## Notes pratiques

- Garder `aphp_generation_utils.py` et le notebook dans le même dossier.
- Après modification du script, réexécuter la première cellule : `importlib.reload(utils)` recharge les fonctions.
- `RANDOM_SELECTION=True` applique d’abord les filtres puis effectue un tirage reproductible avec `RANDOM_SEED`.
- `CLEAN_EXISTING_PROMPTS=True` supprime les anciens `.txt` avant de recréer le manifest ; le passer à `False` avant toute édition manuelle.
- Les sorties individuelles portant les mêmes noms sont écrasées lorsque le même run est relancé.
- Les fichiers JSONL bruts des batchs sont conservés sous `_mistral_batches`.
- Le bilan des tokens et coûts est écrit dans `sorties/mistral_token_usage_and_cost.json`.
- La clé Mistral doit provenir de la variable d’environnement `MISTRAL_API_KEY` et ne doit pas être inscrite dans le notebook.

In [17]:
from pathlib import Path
RUN_DIR = "test_xx/"
# adapte RUN_DIR si besoin : c'est le dossier du run existant
scenarios, manifest = utils.load_state(RUN_DIR)
r = manifest.row(0, named=True)

a = (RUN_DIR / r["one_user_path"]).read_text(encoding="utf-8")
b = (RUN_DIR / r["first_user_path"]).read_text(encoding="utf-8")
c = (RUN_DIR / r["second_user_path"]).read_text(encoding="utf-8")

print("one == first  :", a == b)
print("first == second:", b == c)

TypeError: unsupported operand type(s) for /: 'str' and 'str'